# 3. Graph-EVT: многоканальная POT-детекция

**Цель:** применить `GraphEVTPipeline` к Kuramoto-ряду, оценить POT/GPD-порог на baseline и проверить устойчивое превышение агрегированного индикатора. Для многоканального события библиотека дополнительно оценивает граф зависимостей baseline и ранжирование предполагаемого источника.

In [1]:
from pathlib import Path
from importlib.metadata import version
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_synchronized_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_synchronized_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
assert np.isfinite(x).all() and x.ndim == 2

In [2]:
BASELINE_SIZE = 300
evt_config = EVTConfig(
    baseline_size=BASELINE_SIZE,
    tail_quantile=0.90,
    alarm_probability=0.99,
    top_k=3,
    persistence=2,
)
pipeline = GraphEVTPipeline(
    evt_config,
    graph=GraphConfig(method="ar", edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
    verbose=True,
)
result = pipeline.run_detailed(x)
detection = result.detection

pd.DataFrame([{
    "observations": result.input_profile.n_time,
    "channels": result.input_profile.n_channels,
    "baseline_size": BASELINE_SIZE,
    "alarm_threshold": detection.alarm_threshold,
    "detected": detection.detected,
    "time_index": detection.time_index,
    "graph_available": result.graph is not None,
    "estimated_source": None if result.ranking is None else channel_names[result.ranking.source],
}])

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

,observations,channels,baseline_size,alarm_threshold,detected,time_index,graph_available,estimated_source
0,1500,12,300,5.49498,True,300,True,channel_08


In [3]:
artifact_dir = ROOT / "results/n-d"
artifact_dir.mkdir(parents=True, exist_ok=True)

indicator_frame = pd.DataFrame({
    "time": df["timestamp"],
    "indicator": detection.indicator,
    "above_alarm_threshold": detection.indicator > detection.alarm_threshold,
    "is_baseline": np.arange(len(x)) < BASELINE_SIZE,
})
indicator_frame.to_csv(artifact_dir / "03_evt_indicator.csv", index=False)

provenance = {
    "library": "graph-evt-agent",
    "library_version": version("graph-evt-agent"),
    "input_file": DATA.name,
    "observations": result.input_profile.n_time,
    "channels": result.input_profile.n_channels,
    "seed": SEED,
    "configuration": {
        "baseline_size": BASELINE_SIZE,
        "tail_quantile": evt_config.tail_quantile,
        "alarm_probability": evt_config.alarm_probability,
        "persistence": evt_config.persistence,
        "top_k": evt_config.top_k,
        "graph_method": "ar",
    },
    "detection": {
        "detected": detection.detected,
        "time_index": detection.time_index,
        "alarm_threshold": float(detection.alarm_threshold),
    },
    "graph_available": result.graph is not None,
    "ranking_available": result.ranking is not None,
    "estimated_source": None if result.ranking is None else channel_names[result.ranking.source],
}
with (artifact_dir / "03_evt_provenance.json").open("w", encoding="utf-8") as file:
    json.dump(provenance, file, indent=2)

provenance

{'library': 'graph-evt-agent',
 'library_version': '0.1.0',
 'input_file': 'kuramoto_synchronized_series.csv',
 'observations': 1500,
 'channels': 12,
 'seed': 42,
 'configuration': {'baseline_size': 300,
  'tail_quantile': 0.9,
  'alarm_probability': 0.99,
  'persistence': 2,
  'top_k': 3,
  'graph_method': 'ar'},
 'detection': {'detected': True,
  'time_index': 300,
  'alarm_threshold': 5.494979693764073},
 'graph_available': True,
 'ranking_available': True,
 'estimated_source': 'channel_08'}

Baseline фиксирован до просмотра результата (первые 300 точек), а не выбран по нему. Для многоканального события граф и ranking строятся дополнительно к самому обнаружению; несовпадение оценённого источника с номинальным ведущим каналом из метаданных генератора — ожидаемая ошибка эвристики локализации, а не повод скрыть результат.

**Самопроверка:** почему `top_k=3` усредняет три наибольших отклика, а не берёт только максимум? Как изменится момент тревоги при другом методе графа? Почему граф, оценённый по baseline, не доказывает направление распространения?